# Démo : description des tools

**Contexte :** assistant du service client de Lagune Banque.

Le modèle choisit un tool à partir de son nom, de ses paramètres et de sa description. Cette démo pose les mêmes questions avec deux jeux des mêmes six tools, l'un aux descriptions précises, l'autre aux descriptions vagues, et regarde le tool call qui en sort. Références : [tools](https://docs.langchain.com/oss/python/langchain/tools), [agents](https://docs.langchain.com/oss/python/langchain/agents).

**Sommaire**
- [Configuration](#configuration)
- [Six tools, deux descriptions](#six-tools-deux-descriptions)
- [Le tool call](#le-tool-call)
- [Taux de bon choix](#taux-de-bon-choix)
- [Erreurs](#erreurs)
- [Synthèse](#synthese)

## <a id="configuration"></a>Configuration

In [12]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

from concurrent.futures import ThreadPoolExecutor
from typing import Optional

import pandas as pd
from langchain_core.tools import StructuredTool
from pydantic import BaseModel

from base import AUJOURDHUI, new_model
from config import prepare

In [13]:
prepare()

Connexion OK : gemini/gemini-3.5-flash-lite


## <a id="six-tools-deux-descriptions"></a>Six tools, deux descriptions

Les noms et les paramètres sont les mêmes dans les deux jeux ; seule la description change. Les fonctions ne font rien : on regarde uniquement ce que le modèle décide d'appeler, pas l'exécution.

In [14]:
class Docs(BaseModel):
    requete: str


class Loans(BaseModel):
    pass


class Schedule(BaseModel):
    pret_id: Optional[str] = None


class Transaction(BaseModel):
    date: str
    montant: Optional[int] = None


class Incident(BaseModel):
    date_heure: str


class Claim(BaseModel):
    resume: str
    urgence: str


SCHEMAS = {"search_docs": Docs, "get_customer_loans": Loans, "get_schedule": Schedule, "check_lm_transaction": Transaction,
           "check_platform_incident": Incident, "create_claim": Claim}

PRECISES = {
    "search_docs": "Recherche dans la documentation officielle de Lagune Banque (conditions générales, fiches produits, FAQ, grille tarifaire, procédures). À utiliser pour une question sur les frais, les taux, les conditions ou une procédure.",
    "get_customer_loans": "Liste les prêts du client identifié, leur statut et la suspension éventuelle de l'accès à un nouveau prêt. Ne donne ni échéances ni transactions.",
    "get_schedule": "Échéancier des prêts en cours ou impayés du client identifié : date, montant dû, statut et frais de retard de chaque échéance.",
    "check_lm_transaction": "Recherche les remboursements Lagune Money du client identifié autour d'une date (AAAA-MM-JJ) : statut, code d'échec, échéance rattachée. À utiliser quand le client parle d'un paiement ou d'un virement qu'il a effectué.",
    "check_platform_incident": "Vérifie si un incident technique de la plateforme couvrait une date et une heure précises (AAAA-MM-JJTHH:MM). Ne concerne jamais les transactions d'un client.",
    "create_claim": "Crée une réclamation et transmet le dossier à un conseiller. À utiliser si le client conteste une opération qu'il ne reconnaît pas, ou demande un conseiller.",
}

VAGUES = {
    "search_docs": "Cherche des informations.",
    "get_customer_loans": "Récupère des données du client.",
    "get_schedule": "Récupère des données.",
    "check_lm_transaction": "Vérifie quelque chose.",
    "check_platform_incident": "Vérifie quelque chose.",
    "create_claim": "Enregistre une demande.",
}

def build_tools(descriptions):
    return [StructuredTool.from_function(func=lambda **kw: "{}", name=nom, description=descriptions[nom], args_schema=SCHEMAS[nom])
            for nom in SCHEMAS]


JEUX = {"précises": PRECISES, "vagues": VAGUES}

## <a id="le-tool-call"></a>Le tool call

`ask` envoie la question avec les six tools et renvoie le premier tool call du modèle : le nom du tool et ses arguments, ou `aucun tool` si le modèle répond directement. Le system prompt est volontairement minimal et ne nomme aucun tool.

In [15]:
SYSTEM_PROMPT = f"Tu es l'assistant du service client de Lagune Banque. Nous sommes le {AUJOURDHUI}. Utilise un tool si la question en a besoin."

model = new_model()
LLMS = {nom: model.bind_tools(build_tools(d)) for nom, d in JEUX.items()}


def ask(llm, question):
    return llm.invoke([("system", SYSTEM_PROMPT), ("user", question)]).tool_calls


def first_tool(llm, question):
    calls = ask(llm, question)
    return calls[0]["name"] if calls else None


def show(call):
    return f"{call[0]['name']}({call[0]['args']})" if call else "aucun tool"


EXEMPLES = [
    "Quels sont les frais de retard du Prêt Express ?",
    "Donne-moi le détail de mes échéances et de mes frais de retard.",
    "J'ai remboursé 31 500 FCFA hier via Lagune Money, l'avez-vous bien reçu ?",
    "La plateforme était-elle en panne hier à 15 h ?",
    "Bonjour, merci pour votre aide !",
]
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"question": EXEMPLES, **{nom: [show(ask(llm, q)) for q in EXEMPLES] for nom, llm in LLMS.items()}})

,question,précises,vagues
0,Quels sont les frais de retard du Prêt Express ?,search_docs({'requete': 'Prêt Express frais de retard'}),search_docs({'requete': 'frais de retard Prêt Express'})
1,Donne-moi le détail de mes échéances et de mes frais de retard.,get_schedule({}),get_customer_loans({})
2,"J'ai remboursé 31 500 FCFA hier via Lagune Money, l'avez-vous bien reçu ?","check_lm_transaction({'montant': 31500, 'date': '2026-11-15'})","check_lm_transaction({'date': '2026-11-15', 'montant': 31500})"
3,La plateforme était-elle en panne hier à 15 h ?,check_platform_incident({'date_heure': '2026-11-15T15:00'}),check_platform_incident({'date_heure': '2026-11-15 15:00:00'})
4,"Bonjour, merci pour votre aide !",aucun tool,aucun tool


## <a id="taux-de-bon-choix"></a>Taux de bon choix

17 questions, chacune avec le premier tool attendu, ou `None` quand aucun tool n'est nécessaire. Chaque question est posée `RUNS` fois par jeu.

In [16]:
QUESTIONS = [
    ("Quels sont les frais de retard du Prêt Express ?", "search_docs"),
    ("Quelle est la procédure en cas de suspicion de fraude ?", "search_docs"),
    ("Quel est le taux de l'Épargne Plus ?", "search_docs"),
    ("Quels prêts ai-je en cours ?", "get_customer_loans"),
    ("Mon accès à un nouveau prêt est-il suspendu ?", "get_customer_loans"),
    ("Donne-moi le détail de mes échéances et de mes frais de retard.", "get_schedule"),
    ("Quelle est la date de ma prochaine échéance ?", "get_schedule"),
    ("J'ai remboursé 31 500 FCFA hier via Lagune Money, l'avez-vous bien reçu ?", "check_lm_transaction"),
    ("Mon virement Lagune Money de 25 000 FCFA du 10 novembre est-il passé ?", "check_lm_transaction"),
    ("Y a-t-il eu un incident sur la plateforme le 15 novembre vers 14 h 30 ?", "check_platform_incident"),
    ("La plateforme était-elle en panne hier à 15 h ?", "check_platform_incident"),
    ("Je conteste une opération que je ne reconnais pas sur mon compte.", "create_claim"),
    ("Je veux déposer une réclamation et parler à un conseiller.", "create_claim"),
    ("Mon paiement de 31 500 FCFA du 15 novembre n'est pas rattaché à mon échéance.", "check_lm_transaction"),
    ("Une panne de la plateforme le 15 novembre entre 14 h et 16 h a-t-elle été signalée ?", "check_platform_incident"),
    ("Bonjour, merci pour votre aide !", None),
    ("Merci, c'est tout pour aujourd'hui, au revoir.", None),
]

In [17]:
RUNS = 3


def measure(nom, llm):
    taches = [(q, attendu) for q, attendu in QUESTIONS for _ in range(RUNS)]
    with ThreadPoolExecutor(4) as pool:
        choix = list(pool.map(lambda t: first_tool(llm, t[0]), taches))
    return pd.DataFrame({"jeu": nom, "question": [t[0] for t in taches], "attendu": [t[1] for t in taches], "choisi": choix})


RESULTATS = pd.concat([measure(nom, llm) for nom, llm in LLMS.items()], ignore_index=True)
RESULTATS["bon"] = RESULTATS["attendu"].fillna("aucun") == RESULTATS["choisi"].fillna("aucun")
RESULTATS.groupby("jeu")["bon"].mean().round(2)

jeu
précises    0.98
vagues      0.78
Name: bon, dtype: float64

## <a id="synthese"></a>Synthèse

In [ ]:
n = len(QUESTIONS) * RUNS
TAUX = RESULTATS.groupby("jeu")["bon"].mean()
for jeu, taux in TAUX.items():
    print(f"Descriptions {jeu:9s}: {int(round(taux * n))} bons choix sur {n} ({taux:.0%})")

Descriptions précises : 51 bons choix sur 51 (100%)
Descriptions vagues   : 39 bons choix sur 51 (76%)


À retenir : la description dit à quoi sert le tool, quand l'appeler et ce qu'il ne fait pas. Un nom explicite compense en partie une description vague ; l'écart se creuse là où deux tools se ressemblent (ici les prêts et les échéances).